In [ ]:
%pip install -q ipylab

# Autostart

Autostart is a feature implemented using the [pluggy](https://pluggy.readthedocs.io/en/stable/index.html#pluggy) plugin system. The code associated with the entry point `ipylab_backend` will be called (imported) when `ipylab` is activated. `ipylab` will activate when Jupyterlab is started (provided `ipylab` is installed and enabled). 

There are no limitations to what can be done. But it is recommended to import on demand to minimise the time required to launch.  Some possibilities include:
* Create and register custom commands;
* Create launchers;
* Modify the appearance of Jupyterlab.

## Entry points

Add the following in your `pyproject.toml`

``` toml
[project.entry-points.ipylab_backend]
myproject = "myproject.pluginmodule"
```

In `my_module.autostart.py` write code that will be run once.

Example:

```python
# @ipylab_plugin.py

import asyncio

async def startup():
    import ipylab
    
    app = ipylab.JupyterFrontEnd()  
    await app.read_wait()
    #Do everything to startup

ipylab_plugin = None # Provide an empty object with the expected name.
asyncio.create_task(startup())
```

## Example creating a launcher

In [ ]:
# @my_module.autostart.py

import asyncio

import ipylab


async def create_app(path):
    # The code in this function is called in the new kernel.
    # Ensure imports are performed inside the function.
    import ipywidgets as ipw

    import ipylab

    global ma  # noqa: PLW0603

    ma = ipylab.MainArea(name="My demo app", path=path)
    await ma.wait_ready()
    ma.content.title.label = "Simple app"
    ma.content.title.caption = ma.kernelId
    console_button = ipw.Button(description="Toggle console")
    error_button = ipw.Button(
        description="Do an error",
        tooltip="An error dialog will pop up when this is clicked.\n"
        "The dialog demonstrates the use of the `on_frontend_error` plugin.",
    )
    console_button.on_click(
        lambda _: ma.load_console(name="console") if ma.console_status == "unloaded" else ma.unload_console()
    )
    error_button.on_click(lambda _: ma.executeCommand("Not a command"))
    console_status = ipw.HTML()
    ipw.dlink((ma, "console_status"), (console_status, "value"))
    ma.content.children = [
        ipw.HTML(f"<h3>My simple app</h3> Welcome to my app.<br> kernel id: {ma.kernelId}"),
        ipw.HBox([console_button, error_button]),
        console_status,
    ]

    # Shutdown when MainArea is unloaded.
    def on_status_change(change):
        if change["new"] == "unloaded":
            ma.app.shutdownKernel()

    ma.observe(on_status_change, "status")

    class IpylabPlugins:
        # Define plugins (see IpylabHookspec for available hooks)
        @ipylab.hookimpl
        def on_frontend_error(self, obj, error, content):  # noqa: ARG002
            ma.app.dialog.show_error_message("Error", str(error))

    # Register plugin for this kernel.
    ipylab.hookspecs.pm.register(IpylabPlugins())  # type: ignore

    await ma.load()
    return ipylab.pack(ma)


n = 0
app = ipylab.JupyterFrontEnd()


async def start_my_app(cwd):  # noqa: ARG001
    global n  # noqa: PLW0603
    n += 1
    path = f"my app {n}"
    launcher_id = app.current_widget_id if app.current_widget_id.startswith("launcher") else ""
    await app.execEval(
        code=create_app,
        user_expressions={"main_area_widget": "create_app(path)"},
        path=path,
    )
    if launcher_id:
        await app.executeMethod(widget=launcher_id, method="dispose")


async def register_commands():
    await app.command.addPythonCommand(
        "start_my_app",
        execute=start_my_app,
        label="Start Custom App",
        icon_class="jp-PythonIcon",
    )
    await app.launcher.add_item("start_my_app", "Ipylab")
    return "done"


ipylab_plugin = None
t = asyncio.create_task(register_commands())

In [ ]:
t.result()

In [ ]:
app = ipylab.JupyterFrontEnd()

In [ ]:
t = app.executeCommand("launcher:create")